In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName("twibot")
    .config("spark.jars.packages",
            "graphframes:graphframes:0.8.3-spark3.5-s_2.12,"
            "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0")
    .config("spark.mongodb.write.connection.uri", "mongodb://mongo:27017")
    .config("spark.mongodb.read.connection.uri", "mongodb://mongo:27017")
    .config("spark.driver.memory", "1g")
    .getOrCreate())
out = "/home/jovyan/work/parquet"
print(spark.version)

In [ ]:

for name in ["communities", "community_scores", "user_features"]:
    try:
        n = spark.read.parquet(f"{out}/{name}").count()
        print(name, "read OK:", n, "rows")
    except Exception as e:
        print(name, "read FAILED")
        print([l[:250] for l in str(e).splitlines() if "Caused by" in l][:3])

try:
    t = spark.createDataFrame([("a", 1)], ["k", "v"])
    (t.write.format("mongodb").mode("overwrite")
      .option("database", "twibot").option("collection", "smoke").save())
    print("mongo write OK")
except Exception as e:
    print("mongo write FAILED")
    print([l[:250] for l in str(e).splitlines() if "Caused by" in l][:3])

In [ ]:
def save(df, collection):
    (df.write.format("mongodb").mode("overwrite")
       .option("database", "twibot").option("collection", collection).save())
    print(collection, "saved")

lp = spark.read.parquet(f"{out}/communities")
save(lp.select("id", "bot_label", F.col("label").alias("community")), "user_communities")
save(spark.read.parquet(f"{out}/community_scores"), "community_scores")
save(spark.read.parquet(f"{out}/user_features"), "user_features")
save(spark.read.parquet(f"{out}/community_features"), "community_features")


for c in ["user_communities", "community_scores", "user_features", "community_features"]:
    n = (spark.read.format("mongodb")
         .option("database", "twibot").option("collection", c).load().count())
    print(c, n, "documents")